# 🎬 AI Media Studio — Google Colab Pro L4 Worker v4.0 (Clean-Slate · Zero Drive)

Arquitetura isolada construída do zero para **NVIDIA L4 (24 GB VRAM)** sem montagem do Google Drive:
- **Zero erros de Python 3.13 / pip:** provisiona dois ambientes **Python 3.10 isolados via `uv`** (`envs/seedvc` + `envs/wangp`) com detecção automática da versão CUDA do driver (`cu128` / `cu126` / `cu124`).
- **Zero conflito entre `Seed-VC` e `WanGP`:** `numpy==1.26.4` + `transformers==4.46.3` ficam isolados no worker de voz (`Seed-VC`), enquanto `WanGP` (`shared/api.py`) + FastAPI rodam em `numpy==2.1.2` + `transformers==4.54.0`.
- **Zero caixas vermelhas/avisos falsos:** captura e filtra logs `stderr` barulhentos de `apt`, `git`, `uv`, `hf_hub` e `uvicorn`, mantendo apenas progresso estruturado `[OK]` e falha explícita caso algo real quebre.
- **Sessão persistente `WanGPSession`:** mantém o modelo INT8 escolhido (`LTX-2.5 22B Distilled`, `LongCat Avatar 1.5`, `InfiniteTalk 14B` ou `Wan2.2 I2V 14B`) aquecido entre cortes de cena.


In [ ]:
# @title 1. Limpeza Total & Pré-Requisitos do Sistema (Do Zero · Sem Google Drive)
REBUILD_FROM_SCRATCH = True  # @param {type:'boolean'}

import json, os, re, shutil, subprocess, time
from pathlib import Path

os.environ['PYTHONWARNINGS'] = 'ignore'
os.environ['PIP_NO_WARN_SCRIPT_LOCATION'] = '1'
os.environ['DEBIAN_FRONTEND'] = 'noninteractive'

def run_clean(label, cmd, cwd=None):
    t0 = time.time()
    p = subprocess.run(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    dt = time.time() - t0
    if p.returncode != 0:
        tail = '\n'.join((p.stdout or '').splitlines()[-50:])
        raise RuntimeError(f'[ERRO] {label} falhou (rc={p.returncode}):\n{tail}')
    print(f'[OK] {label} ({dt:.1f}s)')
    return (p.stdout or '').strip()

subprocess.run(['bash', '-lc', 'pkill -f colab_v4_cinema_server || true; pkill -f colab_gpu_server || true; pkill -f cloudflared || true'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)

legacy_dirs = [
    '/content/ai_studio_cache', '/content/venv_cinema311', '/content/seed-vc',
    '/content/WanGP', '/content/LivePortrait', '/content/LatentSync', '/content/MuseTalk',
]
if REBUILD_FROM_SCRATCH:
    legacy_dirs.append('/content/ai_studio_v4')
for ld in legacy_dirs:
    shutil.rmtree(ld, ignore_errors=True)

WS = Path('/content/ai_studio_v4')
for sub in ('repos', 'envs', 'hf_cache', 'models', 'config', 'outputs/wangp', 'tmp'):
    (WS / sub).mkdir(parents=True, exist_ok=True)
print(f'[OK] Workspace limpo pronto em {WS}')

smi_gpu = run_clean('Detecção GPU NVIDIA', [
    'nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'
])
smi_full = subprocess.check_output(['nvidia-smi'], text=True)
m_cuda = re.search(r'CUDA Version:\s*(\d+)\.(\d+)', smi_full)
cuda_major, cuda_minor = (int(m_cuda.group(1)), int(m_cuda.group(2))) if m_cuda else (12, 6)

if (cuda_major, cuda_minor) >= (12, 8):
    cuda_cfg = {'idx': 'cu128', 'torch': '2.10.0', 'tv': '0.25.0', 'ta': '2.10.0', 'install_torchcodec': True}
elif (cuda_major, cuda_minor) >= (12, 6):
    cuda_cfg = {'idx': 'cu126', 'torch': '2.7.1', 'tv': '0.22.1', 'ta': '2.7.1', 'install_torchcodec': False}
else:
    cuda_cfg = {'idx': 'cu124', 'torch': '2.6.0', 'tv': '0.21.0', 'ta': '2.6.0', 'install_torchcodec': False}
cuda_cfg['driver_cuda'] = f'{cuda_major}.{cuda_minor}'
(WS / 'config/cuda_target.json').write_text(json.dumps(cuda_cfg, indent=2), encoding='utf-8')

disk_free_gb = round(shutil.disk_usage('/content').free / (1024 ** 3), 1)
print(f'[INFO] GPU: {smi_gpu} | Driver CUDA: {cuda_major}.{cuda_minor} | PyTorch Alvo: {cuda_cfg["torch"]}+{cuda_cfg["idx"]} | Disco /content livre: {disk_free_gb} GB')

run_clean('apt-get dependências mídia', [
    'bash', '-lc', 'apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1 libportaudio2 libgl1 libglib2.0-0 build-essential'
])
run_clean('Instalação uv (gerenciador rápido em Rust)', [
    'python3', '-m', 'pip', 'install', '-q', '--no-input', '--upgrade', 'uv'
])
run_clean('Download cloudflared CLI', [
    'bash', '-lc',
    'curl -fsSL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared'
])
ff_ver = run_clean('Verificação ffmpeg', ['bash', '-lc', 'ffmpeg -version | head -n 1'])
uv_ver = run_clean('Verificação uv', ['uv', '--version'])
print(f'[OK] Sistema base pronto | {ff_ver} | {uv_ver}')


In [ ]:
# @title 2. Instalação Limpa em Venvs Isolados (Python 3.10.16 · Zero Conflitos)
VIDEO_MODEL = 'ltx2_25_22B_distilled'  # @param ['ltx2_25_22B_distilled', 'longcat_avatar_v1_5', 'infinitetalk', 'i2v_2_2', 'seed_vc_only']
PRELOAD_WEIGHTS = True  # @param {type:'boolean'}
HF_TOKEN = ''  # @param {type:'string'}
COLAB_AUTH_TOKEN = ''  # @param {type:'string'}

import json, os, re, subprocess, time
from pathlib import Path

WS = Path('/content/ai_studio_v4')
cuda_cfg = json.loads((WS / 'config/cuda_target.json').read_text(encoding='utf-8'))

hf_tok = (HF_TOKEN or '').strip()
if not hf_tok:
    try:
        from google.colab import userdata
        hf_tok = (userdata.get('HF_TOKEN') or '').strip()
    except Exception:
        hf_tok = ''

worker_env = {
    'VIDEO_MODEL': VIDEO_MODEL,
    'PRELOAD_WEIGHTS': bool(PRELOAD_WEIGHTS),
    'HF_TOKEN': hf_tok,
    'COLAB_AUTH_TOKEN': (COLAB_AUTH_TOKEN or '').strip(),
}
(WS / 'config/worker_env.json').write_text(json.dumps(worker_env, indent=2), encoding='utf-8')

base_env = {
    **os.environ,
    'PYTHONWARNINGS': 'ignore',
    'PIP_NO_WARN_SCRIPT_LOCATION': '1',
    'UV_NO_PROGRESS': '1',
    'HF_HUB_DISABLE_PROGRESS_BARS': '1',
    'HF_HUB_VERBOSITY': 'error',
    'HF_HUB_ENABLE_HF_TRANSFER': '1',
    'HF_HOME': str(WS / 'hf_cache'),
    'WAN2GP_DISABLE_AUDIO': '1',
    'PYGAME_HIDE_SUPPORT_PROMPT': '1',
    'TF_CPP_MIN_LOG_LEVEL': '3',
}
if hf_tok:
    base_env['HF_TOKEN'] = hf_tok

def run_clean(label, cmd, cwd=None):
    t0 = time.time()
    p = subprocess.run(cmd, cwd=cwd, env=base_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    dt = time.time() - t0
    if p.returncode != 0:
        tail = '\n'.join((p.stdout or '').splitlines()[-60:])
        raise RuntimeError(f'[ERRO] {label} falhou (rc={p.returncode}):\n{tail}')
    print(f'[OK] {label} ({dt:.1f}s)')
    return (p.stdout or '').strip()

run_clean('Provisionamento Python 3.10 (uv)', ['uv', 'python', 'install', '3.10'])

TORCH_VER = cuda_cfg['torch']
TV_VER = cuda_cfg['tv']
TA_VER = cuda_cfg['ta']
TORCH_INDEX = f"https://download.pytorch.org/whl/{cuda_cfg['idx']}"

SEEDVC_VENV = WS / 'envs/seedvc'
SEEDVC_PY = str(SEEDVC_VENV / 'bin/python')
SEEDVC_DIR = WS / 'repos/seed-vc'

if not (SEEDVC_DIR / 'inference.py').is_file():
    run_clean('Clone Plachtaa/seed-vc', ['git', 'clone', '--depth', '1', 'https://github.com/Plachtaa/seed-vc.git', str(SEEDVC_DIR)])

ckpt_link = SEEDVC_DIR / 'checkpoints'
if not ckpt_link.exists() and not ckpt_link.is_symlink():
    ckpt_link.symlink_to(WS / 'hf_cache')

run_clean('Criação venv Python 3.10 Seed-VC', ['uv', 'venv', str(SEEDVC_VENV), '--python', '3.10', '--seed'])
run_clean(f'Instalação PyTorch {TORCH_VER}+{cuda_cfg["idx"]} (Seed-VC)', [
    'uv', 'pip', 'install', '--python', SEEDVC_PY,
    '--index-url', TORCH_INDEX,
    f'torch=={TORCH_VER}', f'torchaudio=={TA_VER}',
])
run_clean('Instalação dependências limpas Seed-VC (zero GUI/portaúdio bloat)', [
    'uv', 'pip', 'install', '--python', SEEDVC_PY,
    'numpy==1.26.4', 'transformers==4.46.3', 'accelerate',
    'huggingface-hub>=0.28.1', 'hf-transfer',
    'scipy==1.13.1', 'librosa==0.10.2', 'munch==4.0.0', 'einops==0.8.0',
    'descript-audio-codec==1.0.0', 'soundfile==0.12.1', 'pydub==0.25.1',
    'pyyaml', 'resampy', 'hydra-core==1.3.2', 'python-dotenv', 'jiwer==3.0.3',
])
svc_check = run_clean('Self-check imports + CUDA Seed-VC', [
    SEEDVC_PY, '-c',
    'import torch, transformers, librosa, munch, soundfile, dac; assert torch.cuda.is_available(); print("SEEDVC_ENV_OK", torch.__version__, torch.cuda.get_device_name(0))'
])
print('   ->', svc_check.splitlines()[-1])

WANGP_VENV = WS / 'envs/wangp'
WANGP_PY = str(WANGP_VENV / 'bin/python')
WANGP_DIR = WS / 'repos/WanGP'

run_clean('Criação venv Python 3.10 Worker/WanGP', ['uv', 'venv', str(WANGP_VENV), '--python', '3.10', '--seed'])
run_clean(f'Instalação PyTorch {TORCH_VER}+{cuda_cfg["idx"]} (Worker/WanGP)', [
    'uv', 'pip', 'install', '--python', WANGP_PY,
    '--index-url', TORCH_INDEX,
    f'torch=={TORCH_VER}', f'torchvision=={TV_VER}', f'torchaudio=={TA_VER}',
])

if VIDEO_MODEL == 'seed_vc_only':
    run_clean('Instalação FastAPI + Uvicorn (seed_vc_only)', [
        'uv', 'pip', 'install', '--python', WANGP_PY,
        'fastapi', 'uvicorn[standard]', 'pydantic>=2.9', 'python-multipart',
    ])
else:
    if not (WANGP_DIR / 'wgp.py').is_file():
        run_clean('Clone deepbeepmeep/WanGP', ['git', 'clone', '--depth', '1', 'https://github.com/deepbeepmeep/WanGP.git', str(WANGP_DIR)])
    ckpts_link = WANGP_DIR / 'ckpts'
    if not ckpts_link.exists() and not ckpts_link.is_symlink():
        ckpts_link.symlink_to(WS / 'models')

    SKIP_PREFIXES = (
        '--extra-index-url',
        'torch', 'torchvision', 'torchaudio', 'numpy',
        'playwright', 'pygame', 'sounddevice', 'freesimplegui',
        'chumpy', 'smplfitter', 'xatlas', 'fast-simplification', 'taichi', 'jax',
        'spacy', 'spacy_pkuseg', 'pycocotools', 'insightface',
        'pyannote.audio', 'audio-separator', 'sherpa-onnx',
        'ddgs', 'pypdf', 'mcp', 'apprise', 'keyring',
    )
    kept_reqs = ['fastapi', 'uvicorn[standard]', 'python-multipart', 'numpy==2.1.2']
    for raw in (WANGP_DIR / 'requirements.txt').read_text(encoding='utf-8').splitlines():
        ln = raw.split('#', 1)[0].strip()
        if not ln:
            continue
        low = ln.lower()
        if 'onnxruntime-gpu' in low and '3.11' in low:
            continue
        if not cuda_cfg.get('install_torchcodec') and (low == 'torchcodec' or low.startswith('torchcodec=')):
            continue
        if any(
            low == s or low.startswith(s + '=') or low.startswith(s + '>') or low.startswith(s + '<')
            or low.startswith(s + '[') or low.startswith(s + ' ') or low.startswith(s + ';') or low.startswith(s + '@')
            for s in SKIP_PREFIXES
        ):
            continue
        kept_reqs.append(ln)

    req_file = WS / 'tmp/wangp_reqs_clean.txt'
    req_file.write_text('\n'.join(kept_reqs) + '\n', encoding='utf-8')
    run_clean(f'Instalação dependências limpas WanGP + FastAPI ({len(kept_reqs)} pacotes)', [
        'uv', 'pip', 'install', '--python', WANGP_PY, '-r', str(req_file),
    ])
    wgp_check = run_clean('Self-check WanGP shared.api.init + CUDA', [
        WANGP_PY, '-c',
        'import torch, fastapi, uvicorn; from pathlib import Path; '
        'import sys; sys.path.insert(0, "/content/ai_studio_v4/repos/WanGP"); '
        'from shared.api import init; '
        's = init(root=Path("/content/ai_studio_v4/repos/WanGP"), config_path=Path("/content/ai_studio_v4/config/wgp_config.json"), output_dir=Path("/content/ai_studio_v4/outputs/wangp"), cli_args=["--attention", "sdpa", "--profile", "4"], console_output=False); '
        'assert torch.cuda.is_available(); '
        'm = s.get_model_metadata("' + VIDEO_MODEL + '"); '
        'print("WANGP_API_OK", torch.__version__, "models=", len(s.list_model_metadata()), "target_model=", bool(m))'
    ])
    print('   ->', wgp_check.splitlines()[-1])

print('✅ Etapa 2 concluída com ZERO conflitos e 100% dos imports verificados na GPU.')


In [ ]:
# @title 3. Pré-Download de Pesos & Auto-Teste Real na L4 GPU (Validação antes do Túnel)
import json, math, os, shutil, struct, subprocess, sys, time, wave
from pathlib import Path

WS = Path('/content/ai_studio_v4')
env_cfg = json.loads((WS / 'config/worker_env.json').read_text(encoding='utf-8'))
VIDEO_MODEL = env_cfg['VIDEO_MODEL']
PRELOAD_WEIGHTS = env_cfg['PRELOAD_WEIGHTS']

SEEDVC_PY = str(WS / 'envs/seedvc/bin/python')
WANGP_PY = str(WS / 'envs/wangp/bin/python')

run_env = {
    **os.environ,
    'PYTHONWARNINGS': 'ignore',
    'HF_HUB_VERBOSITY': 'error',
    'HF_HUB_DISABLE_PROGRESS_BARS': '1',
    'HF_HUB_ENABLE_HF_TRANSFER': '1',
    'HF_HOME': str(WS / 'hf_cache'),
    'WAN2GP_DISABLE_AUDIO': '1',
    'PYGAME_HIDE_SUPPORT_PROMPT': '1',
    'TF_CPP_MIN_LOG_LEVEL': '3',
}
if env_cfg.get('HF_TOKEN'):
    run_env['HF_TOKEN'] = env_cfg['HF_TOKEN']

def write_test_wav(path, freq=220.0, dur=1.2, sr=16000):
    n = int(sr * dur)
    samples = bytearray()
    for i in range(n):
        t = i / sr
        env = math.sin(math.pi * min(1.0, i / 800.0))
        val = int(11000 * env * (0.65 * math.sin(2 * math.pi * freq * t) + 0.35 * math.sin(2 * math.pi * freq * 2.01 * t)))
        samples.extend(struct.pack('<h', val))
    with wave.open(str(path), 'wb') as wf:
        wf.setnchannels(1); wf.setsampwidth(2); wf.setframerate(sr); wf.writeframes(bytes(samples))

t0 = time.time()
p_cuda = subprocess.run([
    WANGP_PY, '-c',
    'import torch; x = torch.randn((2048, 2048), device="cuda", dtype=torch.float16); y = x @ x; torch.cuda.synchronize(); '
    'free_b, tot_b = torch.cuda.mem_get_info(0); print(f"CUDA_L4_OK {torch.cuda.get_device_name(0)} VRAM_livre={free_b/1024**3:.1f}/{tot_b/1024**3:.1f}GB")'
], env=run_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
if p_cuda.returncode != 0:
    raise RuntimeError(p_cuda.stdout)
print(f'[OK] {p_cuda.stdout.strip()} ({time.time()-t0:.2f}s)')

if not PRELOAD_WEIGHTS:
    print('[INFO] PRELOAD_WEIGHTS=False: pesos serão baixados sob demanda no primeiro job da VPS.')
else:
    smoke_dir = WS / 'tmp/smoke_v4'
    smoke_dir.mkdir(parents=True, exist_ok=True)
    src_wav = smoke_dir / 'src.wav'
    tgt_wav = smoke_dir / 'tgt.wav'
    svc_out = smoke_dir / 'svc_out'
    svc_out.mkdir(exist_ok=True)
    write_test_wav(src_wav, freq=210.0, dur=1.2)
    write_test_wav(tgt_wav, freq=145.0, dur=1.6)

    t1 = time.time()
    print('⏳ Baixando pesos e validando Seed-VC (--f0-condition False) na L4 GPU...')
    p_svc = subprocess.run([
        SEEDVC_PY, str(WS / 'repos/seed-vc/inference.py'),
        '--source', str(src_wav),
        '--target', str(tgt_wav),
        '--output', str(svc_out),
        '--diffusion-steps', '10',
        '--f0-condition', 'False',
        '--auto-f0-adjust', 'True',
        '--fp16', 'True',
    ], cwd=str(WS / 'repos/seed-vc'), env=run_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, timeout=600)
    out_wavs = list(svc_out.glob('*.wav'))
    if p_svc.returncode != 0 or not out_wavs:
        raise RuntimeError(f'Seed-VC auto-teste falhou:\n{(p_svc.stdout or "")[-1500:]}')
    print(f'[OK] Seed-VC L4 validado com sucesso ({time.time()-t1:.1f}s) -> {out_wavs[0].name} ({out_wavs[0].stat().st_size//1024} KB)')

    if VIDEO_MODEL != 'seed_vc_only':
        t2 = time.time()
        print(f'⏳ Baixando pesos INT8 e validando micro-render "{VIDEO_MODEL}" via WanGP na L4 (1ª execução pode levar alguns minutos)...')
        smoke_img = smoke_dir / 'frame.png'
        subprocess.run([
            'ffmpeg', '-y', '-f', 'lavfi', '-i', 'color=c=#2a2018:s=480x832', '-frames:v', '1', str(smoke_img)
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)

        smoke_script = (
            'import sys\n'
            'from pathlib import Path\n'
            'sys.path.insert(0, "/content/ai_studio_v4/repos/WanGP")\n'
            'from shared.api import init\n'
            's = init(\n'
            '    root=Path("/content/ai_studio_v4/repos/WanGP"),\n'
            '    config_path=Path("/content/ai_studio_v4/config/wgp_config.json"),\n'
            '    output_dir=Path("/content/ai_studio_v4/tmp/smoke_v4/wgp_out"),\n'
            '    cli_args=["--attention", "sdpa", "--profile", "4"],\n'
            '    console_output=False,\n'
            ')\n'
            f'model_id = "{VIDEO_MODEL}"\n'
            'cfg = s.get_default_settings(model_id)\n'
            'mdef = s.get_model_def(model_id) or {}\n'
            'schema = (s.get_model_schema(model_id) or {}).get("metadata", {})\n'
            'f_min = int(schema.get("frames_minimum") or mdef.get("frames_minimum") or 17)\n'
            'res = "384x640" if "ltx2" in model_id else "480x832"\n'
            'cfg["model_type"] = model_id\n'
            'cfg["prompt"] = "Cinematic still woman standing in warm candlelit banquet hall, subtle breathing motion."\n'
            'cfg["negative_prompt"] = "watermark, overlaid text"\n'
            'cfg["resolution"] = res\n'
            'cfg["video_length"] = f_min\n'
            f'cfg["image_start"] = "{smoke_img}"\n'
            'cfg["image_prompt_type"] = "S"\n'
            'if model_id in ("infinitetalk", "longcat_avatar_v1_5") or "ltx2" in model_id:\n'
            f'    cfg["audio_guide"] = "{tgt_wav}"\n'
            '    cfg["audio_prompt_type"] = "AS" if "ltx2" in model_id else "A"\n'
            'cfg["num_inference_steps"] = 8 if ("ltx2" in model_id or "longcat" in model_id) else 6\n'
            'cfg["overrides"] = "attention_mode=\'sdpa\', memory_profile=4, transformer_quantization=\'int8\'"\n'
            'job = s.submit_task(cfg)\n'
            'r = job.result()\n'
            'assert r.success and r.generated_files, "WanGP falhou: " + "; ".join(str(e) for e in r.errors)\n'
            'p = Path(r.generated_files[0])\n'
            'print("WANGP_SMOKE_OK", model_id, res, "frames=", f_min, "bytes=", p.stat().st_size)\n'
        )
        smoke_py_file = smoke_dir / 'run_smoke_wgp.py'
        smoke_py_file.write_text(smoke_script, encoding='utf-8')
        p_wgp = subprocess.run([WANGP_PY, str(smoke_py_file)], env=run_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, timeout=1200)
        if p_wgp.returncode != 0 or 'WANGP_SMOKE_OK' not in (p_wgp.stdout or ''):
            tail = '\n'.join((p_wgp.stdout or '').splitlines()[-60:])
            raise RuntimeError(f'WanGP auto-teste "{VIDEO_MODEL}" falhou:\n{tail}')
        print(f'[OK] {p_wgp.stdout.strip().splitlines()[-1]} ({time.time()-t2:.1f}s)')

    shutil.rmtree(smoke_dir, ignore_errors=True)
    disk_after = round(shutil.disk_usage('/content').free / (1024 ** 3), 1)
    print(f'✅ Pré-download + validação GPU concluídos! Disco /content livre: {disk_after} GB')


In [ ]:
# @title 4. Iniciar Servidor Worker Cinema Engine v4.0 + Túnel Cloudflare
import json, os, re, subprocess, time, urllib.request
from pathlib import Path

WS = Path('/content/ai_studio_v4')
env_cfg = json.loads((WS / 'config/worker_env.json').read_text(encoding='utf-8'))

SERVER_CODE = r'''
#!/usr/bin/env python3
"""Google Colab Pro L4 (24GB VRAM) Cinema Engine v4.0 Worker & Benchmark Server.

Zero-Drive, Clean-Slate BigTech Architecture:
1. Local high-speed NVMe runtime under /content/ai_studio_v4 (zero Google Drive FUSE slowdowns/popups).
2. Dual isolated uv Python 3.10 virtual environments:
   - /content/ai_studio_v4/envs/seedvc : isolated numpy==1.26.4 + transformers==4.46.3 Seed-VC CLI runtime.
   - /content/ai_studio_v4/envs/wangp  : isolated FastAPI + WanGP shared/api.py persistent session runtime.
3. Persistent in-process WanGPSession (no cold 14B-22B reloads between shots) with INT8/SDPA/Profile-4 L4 memory safety.
4. Honest /health telemetry (GPU name, real free/total VRAM, disk free GB, loaded model & readiness flags).
5. Async Job Queue (/jobs/animate_scene + /jobs/{id}) immune to Cloudflare 524 100s timeouts.
6. Clean stdout logging (zero noisy stderr red/pink boxes in Colab outputs).
"""

from __future__ import annotations

import base64
import gc
import hmac
import json
import math
import os
import shutil
import subprocess
import sys
import threading
import time
import uuid
from pathlib import Path
from typing import Any, Dict, Optional

import torch
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
import uvicorn


os.environ.setdefault("WAN2GP_DISABLE_AUDIO", "1")
os.environ.setdefault("PYGAME_HIDE_SUPPORT_PROMPT", "1")
os.environ.setdefault("HF_HUB_VERBOSITY", "error")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

WORKSPACE_ROOT = Path(os.environ.get("AI_STUDIO_ROOT", "/content/ai_studio_v4"))
REPOS_DIR = WORKSPACE_ROOT / "repos"
HF_CACHE_DIR = Path(os.environ.get("HF_HOME", str(WORKSPACE_ROOT / "hf_cache")))
CONFIG_DIR = WORKSPACE_ROOT / "config"
OUTPUTS_DIR = WORKSPACE_ROOT / "outputs" / "wangp"

COLAB_AUTH_TOKEN = os.environ.get("COLAB_AUTH_TOKEN", "").strip()
DEFAULT_VIDEO_MODEL = os.environ.get("AI_STUDIO_VIDEO_MODEL", "ltx2_25_22B_distilled").strip()

SEED_VC_PYTHON = Path(os.environ.get("AI_STUDIO_SEEDVC_PY", str(WORKSPACE_ROOT / "envs" / "seedvc" / "bin" / "python")))
SEED_VC_DIR = Path(os.environ.get("AI_STUDIO_SEEDVC_DIR", str(REPOS_DIR / "seed-vc")))
WANGP_DIR = Path(os.environ.get("AI_STUDIO_WANGP_DIR", str(REPOS_DIR / "WanGP")))

GPU_LOCK = threading.Lock()
JOBS_LOCK = threading.Lock()
JOBS: Dict[str, Dict[str, Any]] = {}
_WANGP_SESSION = None


BACKEND_MODEL_MAP: Dict[str, str] = {
    "auto": DEFAULT_VIDEO_MODEL,
    "wangp_wan22_s2v": DEFAULT_VIDEO_MODEL,
    "wan22_s2v_v4": DEFAULT_VIDEO_MODEL,
    "ltx25_v4": "ltx2_25_22B_distilled",
    "ltx25_distilled": "ltx2_25_22B_distilled",
    "ltx2_25_22b_distilled": "ltx2_25_22B_distilled",
    "longcat_avatar": "longcat_avatar_v1_5",
    "longcat_avatar_v1_5": "longcat_avatar_v1_5",
    "infinitetalk": "infinitetalk",
    "multitalk": "multitalk",
    "i2v_2_2": "i2v_2_2",
}


def _cleanup_vram() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


def _gpu_info() -> Dict[str, Any]:
    if not torch.cuda.is_available():
        return {"gpu": "CPU", "vram_gb": 0.0, "vram_free_gb": 0.0, "cuda_ok": False}
    props = torch.cuda.get_device_properties(0)
    free_b, total_b = torch.cuda.mem_get_info(0)
    return {
        "gpu": torch.cuda.get_device_name(0),
        "vram_gb": round(total_b / (1024 ** 3), 1),
        "vram_free_gb": round(free_b / (1024 ** 3), 1),
        "cuda_ok": True,
        "torch_version": torch.__version__,
        "cuda_version": getattr(torch.version, "cuda", ""),
    }


def _smi_used_gb() -> float:
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
            text=True,
            timeout=5,
        ).strip().splitlines()
        if out:
            return round(float(out[0].strip()) / 1024.0, 2)
    except Exception:
        pass
    if torch.cuda.is_available():
        return round(torch.cuda.max_memory_allocated() / (1024 ** 3), 2)
    return 0.0


def _require_token(token: str, require_configured: bool = False) -> None:
    if require_configured and not COLAB_AUTH_TOKEN:
        raise HTTPException(status_code=403, detail="Rota administrativa exige COLAB_AUTH_TOKEN configurado.")
    if COLAB_AUTH_TOKEN and not hmac.compare_digest(token or "", COLAB_AUTH_TOKEN):
        raise HTTPException(status_code=401, detail="X-Colab-Token inválido")


def _probe_dur(path: Path) -> float:
    out = subprocess.check_output(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", str(path)],
        text=True,
    ).strip()
    return max(0.3, float(out))


def _mux_no_freeze(raw_mp4: Path, wav_path: Optional[Path], out_mp4: Path, width: int, height: int, fps: int = 24) -> None:
    vf = f"scale={width}:{height}:force_original_aspect_ratio=increase:flags=lanczos,crop={width}:{height},setsar=1,fps={fps},format=yuv420p"
    vid_dur = _probe_dur(raw_mp4)
    if wav_path and wav_path.is_file():
        aud_dur = _probe_dur(wav_path)
        target_dur = min(vid_dur, aud_dur)
        cmd = [
            "ffmpeg", "-y", "-i", str(raw_mp4), "-i", str(wav_path),
            "-map", "0:v:0", "-map", "1:a:0",
            "-vf", vf, "-t", f"{target_dur:.3f}",
            "-c:v", "libx264", "-preset", "fast", "-crf", "17",
            "-c:a", "aac", "-b:a", "192k", "-ar", "48000", "-ac", "2",
            str(out_mp4),
        ]
    else:
        cmd = [
            "ffmpeg", "-y", "-i", str(raw_mp4),
            "-vf", vf, "-c:v", "libx264", "-preset", "fast", "-crf", "17",
            "-c:a", "aac", "-b:a", "192k", "-ar", "48000", "-ac", "2",
            str(out_mp4),
        ]
    subprocess.run(cmd, check=True, capture_output=True)


def _get_wangp_session():
    global _WANGP_SESSION
    if _WANGP_SESSION is not None:
        return _WANGP_SESSION
    if not (WANGP_DIR / "shared" / "api.py").is_file():
        raise RuntimeError(f"WanGP não instalado em {WANGP_DIR}")
    CONFIG_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
    if str(WANGP_DIR) not in sys.path:
        sys.path.insert(0, str(WANGP_DIR))
    from shared.api import init as wangp_init  # type: ignore

    _WANGP_SESSION = wangp_init(
        root=WANGP_DIR,
        config_path=CONFIG_DIR / "wgp_config.json",
        output_dir=OUTPUTS_DIR,
        cli_args=["--attention", "sdpa", "--profile", "4"],
        console_output=False,
    )
    return _WANGP_SESSION


def _resolve_model_type(requested_backend: str, has_audio: bool) -> str:
    key = (requested_backend or "auto").strip().lower()
    if key == "seed_vc_only":
        raise RuntimeError("Este worker foi iniciado em modo seed_vc_only (sem modelo de vídeo).")
    resolved = BACKEND_MODEL_MAP.get(key) or DEFAULT_VIDEO_MODEL
    if resolved == "seed_vc_only":
        raise RuntimeError("Worker configurado apenas para Seed-VC (VIDEO_MODEL=seed_vc_only).")
    if not has_audio and resolved in ("infinitetalk", "multitalk", "longcat_avatar_v1_5"):
        return "i2v_2_2"
    return resolved


def _resolve_model_resolution(model_type: str, width: int, height: int) -> str:
    portrait = height >= width
    if model_type in ("infinitetalk", "multitalk"):
        return "480x832" if portrait else "832x480"
    if "ltx2" in model_type:
        w = max(256, (width // 32) * 32)
        h = max(256, (height // 32) * 32)
        return f"{w}x{h}"
    if "longcat" in model_type:
        return "480x832" if portrait else "832x480"
    return "480x832" if portrait else "832x480"


app = FastAPI(title="AI Media Studio Colab Cinema Engine v4.0", version="4.0.0-clean-l4")


class ExecReq(BaseModel):
    cmd: str
    timeout: int = 600


class VoiceConvertReq(BaseModel):
    guide_audio_b64: str
    target_timbre_b64: str
    character_id: str = "dona_socorro"
    diffusion_steps: int = 25
    semi_tone_shift: int = 0
    f0_condition: bool = False


class AnimateSceneReq(BaseModel):
    image_b64: str
    audio_b64: str = ""
    prompt: str
    negative_prompt: str = "static portrait, frozen posture, distorted hands, extra fingers, watermark, overlaid text"
    backend: str = "auto"
    steps: int = 8
    guidance_scale: float = 3.5
    duration_sec: float = 3.2
    width: int = 720
    height: int = 1280
    fps: int = 24
    seed: int = 42


@app.get("/health")
def health(x_colab_token: str = Header(default="")):
    _require_token(x_colab_token, require_configured=False)
    info = _gpu_info()
    disk_free_gb = 0.0
    try:
        st = shutil.disk_usage(str(WORKSPACE_ROOT if WORKSPACE_ROOT.exists() else Path("/content")))
        disk_free_gb = round(st.free / (1024 ** 3), 1)
    except Exception:
        pass
    wangp_ready = (WANGP_DIR / "wgp.py").is_file() and (WANGP_DIR / "shared" / "api.py").is_file()
    return {
        "ok": True,
        "ready": True,
        "version": "4.0.0-clean-l4",
        **info,
        "default_video_model": DEFAULT_VIDEO_MODEL,
        "workspace_root": str(WORKSPACE_ROOT),
        "cache_free_gb": disk_free_gb,
        "drive_mounted": False,
        "seed_vc_ready": SEED_VC_PYTHON.is_file() and (SEED_VC_DIR / "inference.py").is_file(),
        "wangp_ready": wangp_ready and DEFAULT_VIDEO_MODEL != "seed_vc_only",
        "ltx2_ready": wangp_ready and "ltx2" in DEFAULT_VIDEO_MODEL,
        "longcat_ready": wangp_ready and "longcat" in DEFAULT_VIDEO_MODEL,
        "wangp_session_warm": _WANGP_SESSION is not None,
        "active_jobs": sum(1 for j in JOBS.values() if j.get("status") == "processing"),
    }


@app.post("/admin/exec")
def admin_exec(req: ExecReq, x_colab_token: str = Header(default="")):
    _require_token(x_colab_token, require_configured=True)
    proc = subprocess.run(req.cmd, shell=True, capture_output=True, text=True, timeout=req.timeout)
    return {"rc": proc.returncode, "stdout": proc.stdout[-60000:], "stderr": proc.stderr[-25000:]}


@app.post("/voice-convert")
def voice_convert(req: VoiceConvertReq, x_colab_token: str = Header(default="")):
    _require_token(x_colab_token, require_configured=False)
    inf_py = SEED_VC_DIR / "inference.py"
    if not inf_py.is_file() or not SEED_VC_PYTHON.is_file():
        raise HTTPException(status_code=503, detail=f"Seed-VC não encontrado em {SEED_VC_DIR}")

    job_dir = WORKSPACE_ROOT / "tmp" / f"svc_v4_{uuid.uuid4().hex[:8]}"
    job_dir.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    with GPU_LOCK:
        try:
            g_wav = job_dir / "source.wav"
            t_wav = job_dir / "target.wav"
            out_dir = job_dir / "out"
            out_dir.mkdir(exist_ok=True)
            out_wav = job_dir / "converted_48k.wav"
            g_wav.write_bytes(base64.b64decode(req.guide_audio_b64))
            t_wav.write_bytes(base64.b64decode(req.target_timbre_b64))

            cmd = [
                str(SEED_VC_PYTHON), str(inf_py),
                "--source", str(g_wav),
                "--target", str(t_wav),
                "--output", str(out_dir),
                "--diffusion-steps", str(int(req.diffusion_steps)),
                "--semi-tone-shift", str(int(req.semi_tone_shift)),
                "--f0-condition", "True" if req.f0_condition else "False",
                "--auto-f0-adjust", "True" if not req.f0_condition else "False",
                "--fp16", "True",
            ]
            env = {
                **os.environ,
                "HF_HOME": str(HF_CACHE_DIR),
                "PYTHONWARNINGS": "ignore",
                "HF_HUB_VERBOSITY": "error",
            }
            proc = subprocess.run(cmd, cwd=str(SEED_VC_DIR), env=env, capture_output=True, text=True, timeout=180)
            gen = next(iter(sorted(out_dir.glob("*.wav"))), None)
            if proc.returncode != 0 or not gen or not gen.is_file():
                raise RuntimeError(f"Seed-VC falhou (rc={proc.returncode}): {(proc.stderr or proc.stdout)[-400:]}")

            subprocess.run(
                ["ffmpeg", "-y", "-i", str(gen), "-ar", "48000", "-ac", "2", "-c:a", "pcm_s16le", str(out_wav)],
                check=True,
                capture_output=True,
            )
            return {
                "ok": True,
                "engine": "seed-vc-v4-real",
                "elapsed_s": round(time.time() - t0, 2),
                "duration": round(_probe_dur(out_wav), 2),
                "vram_used_gb": _smi_used_gb(),
                "character_id": req.character_id,
                "audio_b64": base64.b64encode(out_wav.read_bytes()).decode("ascii"),
            }
        except Exception as exc:
            raise HTTPException(status_code=500, detail=str(exc)[:500])
        finally:
            shutil.rmtree(job_dir, ignore_errors=True)
            _cleanup_vram()


def _execute_scene_job(job_id: str, req: AnimateSceneReq) -> None:
    job_dir = WORKSPACE_ROOT / "tmp" / f"scene_v4_{job_id}"
    job_dir.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    try:
        with JOBS_LOCK:
            JOBS[job_id]["status"] = "processing"
        img_p = job_dir / "keyframe.jpg"
        img_p.write_bytes(base64.b64decode(req.image_b64))
        wav_p: Optional[Path] = None
        dur_target = float(req.duration_sec or 3.2)
        if req.audio_b64:
            wav_p = job_dir / "dialogue.wav"
            wav_p.write_bytes(base64.b64decode(req.audio_b64))
            dur_target = max(1.0, min(10.0, _probe_dur(wav_p)))

        fps = int(req.fps or 24)
        final_mp4 = job_dir / "final_scene.mp4"
        resolved_model = _resolve_model_type(req.backend, has_audio=(wav_p is not None))

        with GPU_LOCK:
            session = _get_wangp_session()
            settings = session.get_default_settings(resolved_model)
            mdef = session.get_model_def(resolved_model) or {}
            schema = (session.get_model_schema(resolved_model) or {}).get("metadata", {})
            f_min = int(schema.get("frames_minimum") or mdef.get("frames_minimum") or 17)
            f_step = int(schema.get("frames_steps") or mdef.get("frames_steps") or 8)
            f_off = int(mdef.get("frames_offset") or 1)
            raw_frames = max(f_min, int(math.ceil(dur_target * fps)) + 4)
            steps_n = max(0, (raw_frames - f_off + f_step - 1) // f_step)
            aligned_frames = max(f_min, steps_n * f_step + f_off)

            settings["model_type"] = resolved_model
            settings["prompt"] = req.prompt
            settings["negative_prompt"] = req.negative_prompt
            settings["resolution"] = _resolve_model_resolution(resolved_model, int(req.width), int(req.height))
            settings["video_length"] = aligned_frames
            settings["force_fps"] = fps
            settings["seed"] = int(req.seed)
            settings["image_start"] = str(img_p)
            settings["image_prompt_type"] = "S"
            if wav_p is not None:
                settings["audio_guide"] = str(wav_p)
                settings["audio_prompt_type"] = "AS" if "ltx2" in resolved_model else "A"
            if "ltx2" in resolved_model and "distilled" in resolved_model:
                settings["num_inference_steps"] = 8
            elif "longcat" in resolved_model:
                settings["num_inference_steps"] = 8
                settings["guidance_scale"] = 1.0
                settings["audio_guidance_scale"] = 1.0
            else:
                settings["num_inference_steps"] = max(4, int(req.steps))
                settings["guidance_scale"] = float(req.guidance_scale)
            settings["overrides"] = "attention_mode='sdpa', memory_profile=4, transformer_quantization='int8'"

            job = session.submit_task(settings)
            result = job.result()
            if not result.success or not result.generated_files:
                errs = "; ".join(str(e) for e in (result.errors or [])) or "WanGP retornou 0 arquivos."
                raise RuntimeError(errs[:500])

            raw_cand = Path(result.generated_files[0])
            _mux_no_freeze(raw_cand, wav_p, final_mp4, int(req.width), int(req.height), fps=fps)
            peak_vram = _smi_used_gb()
            _cleanup_vram()

        vid_b64 = base64.b64encode(final_mp4.read_bytes()).decode("ascii")
        with JOBS_LOCK:
            JOBS[job_id].update({
                "status": "completed",
                "ok": True,
                "backend": resolved_model,
                "elapsed_s": round(time.time() - t0, 2),
                "duration": round(_probe_dur(final_mp4), 2),
                "vram_peak_gb": peak_vram,
                "video_b64": vid_b64,
            })
    except Exception as exc:
        with JOBS_LOCK:
            JOBS[job_id].update({
                "status": "error",
                "ok": False,
                "elapsed_s": round(time.time() - t0, 2),
                "error": str(exc)[:600],
            })
    finally:
        shutil.rmtree(job_dir, ignore_errors=True)
        _cleanup_vram()


@app.post("/jobs/animate_scene")
def submit_scene_job(req: AnimateSceneReq, x_colab_token: str = Header(default="")):
    _require_token(x_colab_token, require_configured=False)
    job_id = uuid.uuid4().hex[:10]
    with JOBS_LOCK:
        if len(JOBS) > 40:
            for k in list(JOBS.keys())[:20]:
                JOBS.pop(k, None)
        JOBS[job_id] = {"ok": True, "job_id": job_id, "status": "queued", "created_at": time.time()}
    threading.Thread(target=_execute_scene_job, args=(job_id, req), daemon=True).start()
    return {"ok": True, "job_id": job_id, "status": "queued"}


@app.get("/jobs/{job_id}")
def get_scene_job(job_id: str, x_colab_token: str = Header(default="")):
    _require_token(x_colab_token, require_configured=False)
    with JOBS_LOCK:
        job = JOBS.get(job_id)
    if not job:
        raise HTTPException(status_code=404, detail="Job não encontrado")
    return job


if __name__ == "__main__":
    log_cfg = {
        "version": 1,
        "disable_existing_loggers": False,
        "formatters": {
            "default": {"format": "%(asctime)s [%(levelname)s] %(message)s"},
        },
        "handlers": {
            "file": {
                "class": "logging.FileHandler",
                "filename": "/tmp/colab_v4_server.log",
                "formatter": "default",
            },
            "stdout": {
                "class": "logging.StreamHandler",
                "stream": "ext://sys.stdout",
                "formatter": "default",
            },
        },
        "loggers": {
            "uvicorn": {"handlers": ["file", "stdout"], "level": "WARNING", "propagate": False},
            "uvicorn.error": {"handlers": ["file", "stdout"], "level": "WARNING", "propagate": False},
            "uvicorn.access": {"handlers": ["file"], "level": "INFO", "propagate": False},
        },
    }
    uvicorn.run(app, host="127.0.0.1", port=int(os.environ.get("PORT", "8787")), log_config=log_cfg)
'''

srv_file = WS / 'colab_v4_cinema_server.py'
srv_file.write_text(SERVER_CODE.lstrip('\n'), encoding='utf-8')

subprocess.run(['bash', '-lc', 'pkill -f colab_v4_cinema_server || true; pkill -f cloudflared || true'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)

srv_env = {
    **os.environ,
    'PORT': '8787',
    'AI_STUDIO_ROOT': str(WS),
    'AI_STUDIO_VIDEO_MODEL': env_cfg['VIDEO_MODEL'],
    'COLAB_AUTH_TOKEN': env_cfg.get('COLAB_AUTH_TOKEN') or '',
    'HF_HOME': str(WS / 'hf_cache'),
    'PYTHONWARNINGS': 'ignore',
    'HF_HUB_VERBOSITY': 'error',
    'HF_HUB_DISABLE_PROGRESS_BARS': '1',
    'HF_HUB_ENABLE_HF_TRANSFER': '1',
    'WAN2GP_DISABLE_AUDIO': '1',
    'PYGAME_HIDE_SUPPORT_PROMPT': '1',
    'TF_CPP_MIN_LOG_LEVEL': '3',
}
if env_cfg.get('HF_TOKEN'):
    srv_env['HF_TOKEN'] = env_cfg['HF_TOKEN']

py_bin = str(WS / 'envs/wangp/bin/python')
srv_log = open('/tmp/colab_v4_server.log', 'w', encoding='utf-8')
srv_proc = subprocess.Popen(
    [py_bin, str(srv_file)],
    env=srv_env,
    stdout=srv_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

hdrs = {'X-Colab-Token': srv_env['COLAB_AUTH_TOKEN']} if srv_env['COLAB_AUTH_TOKEN'] else {}
health_data = None
for _ in range(40):
    if srv_proc.poll() is not None:
        tail = Path('/tmp/colab_v4_server.log').read_text(encoding='utf-8', errors='ignore')[-2000:]
        raise RuntimeError(f'Servidor FastAPI encerrou prematuramente:\n{tail}')
    try:
        req = urllib.request.Request('http://127.0.0.1:8787/health', headers=hdrs)
        with urllib.request.urlopen(req, timeout=2) as r:
            health_data = json.loads(r.read().decode('utf-8'))
            if health_data.get('ok'):
                break
    except Exception:
        time.sleep(0.5)
if not health_data:
    raise RuntimeError('Timeout aguardando http://127.0.0.1:8787/health')

print(f"[OK] Worker FastAPI Cinema v4.0 ativo | GPU={health_data.get('gpu')} | VRAM={health_data.get('vram_free_gb')}/{health_data.get('vram_gb')}GB | model={health_data.get('default_video_model')}")

cf_log = open('/tmp/cloudflared_v4.log', 'w', encoding='utf-8')
cf_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8787'],
    stdout=cf_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

tunnel_url = ''
for _ in range(60):
    txt = Path('/tmp/cloudflared_v4.log').read_text(encoding='utf-8', errors='ignore')
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', txt)
    if m:
        tunnel_url = m.group(0)
        break
    time.sleep(0.5)

if not tunnel_url:
    raise RuntimeError('Não foi possível obter URL trycloudflare.com em 30s.')

print('\n====================================================================')
print('🚀 PONTE L4 v4.0 PRONTA — COLE NO PAINEL DA VPS OU ENVIE NO CHAT:')
print(tunnel_url)
print('====================================================================\n', flush=True)

while True:
    time.sleep(60)
    if srv_proc.poll() is not None or cf_proc.poll() is not None:
        print('⚠️ Processo servidor/túnel encerrou; verifique /tmp/colab_v4_server.log')
        break
    try:
        req = urllib.request.Request('http://127.0.0.1:8787/health', headers=hdrs)
        with urllib.request.urlopen(req, timeout=3) as r:
            h = json.loads(r.read().decode('utf-8'))
        print(f"💚 Heartbeat {time.strftime('%H:%M:%S')} | VRAM Livre: {h.get('vram_free_gb')}/{h.get('vram_gb')}GB | Disco Livre: {h.get('cache_free_gb')}GB | Jobs: {h.get('active_jobs')}", flush=True)
    except Exception:
        pass
